In [2]:
import sqlite3
import pandas as pd
from pathlib import Path 

In [3]:
# Загружаю данные в базу данных sqlite, чтобы писать sql-запросы

csv_files = list(Path("data/input").glob("*.csv"))
connection = sqlite3.connect("olist.db")

for file in csv_files:
    table_name = file.name.replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")
    df = pd.read_csv(file)
    df.to_sql(table_name, connection, if_exists="replace", index=False)
connection.close()

### Сбор витрины

In [11]:
connection = sqlite3.connect("olist.db")

# Рассчитываю витрину, которая ляжет в основу дашборда
query = """
select
    o.order_id,
    c.customer_unique_id,
    row_number() over (partition by c.customer_unique_id order by  o.order_purchase_timestamp) as customer_order_number,
    c.customer_city,
    c.customer_state,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    round(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp), 0) as delivery_days,
    strftime('%Y-%m', o.order_purchase_timestamp) as month_date,
    strftime('%Y', o.order_purchase_timestamp) as year_date,
    sum(oi.price) as order_total,
    sum(oi.freight_value) as freight_total,
    count(oi.order_item_id) as items_count
from orders o
join customers c on o.customer_id = c.customer_id
join order_items oi on o.order_id = oi.order_id
group by 
    o.order_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date
;
"""

# Получаю данные и сохраняю их в датафрейм
mart = pd.read_sql(query, connection)
connection.close()

# Сохраняю витрину в csv файл, чтобы использовать ее в DataLens
mart.to_csv("data/output/olist_mart.csv", index=False)

mart


,order_id,customer_unique_id,customer_order_number,customer_city,customer_state,order_purchase_timestamp,order_delivered_customer_date,delivery_days,month_date,year_date,order_total,freight_total,items_count
0,e22acc9c116caa3f2b7121bbb380d08e,0000366f3b9a7992bf8c76cfdf3221e2,1,cajamar,SP,2018-05-10 10:56:27,2018-05-16 20:48:37,6.0,2018-05,2018,129.90,12.00,1
1,3594e05a005ac4d06a72673270ef9ec9,0000b849f77a49e4a4ce2b2a4ca5be3f,1,osasco,SP,2018-05-07 11:11:27,2018-05-10 18:02:42,3.0,2018-05,2018,18.90,8.29,1
2,b33ec3b699337181488304f362a6b734,0000f46a3911fa3c0805444483337064,1,sao jose,SC,2017-03-10 21:05:03,2017-04-05 14:38:47,26.0,2017-03,2017,69.00,17.22,1
3,41272756ecddd9a9ed0180413cc22fb6,0000f6ccb0745a6a4b88665a16c9f078,1,belem,PA,2017-10-12 20:29:41,2017-11-01 21:23:05,20.0,2017-10,2017,25.99,17.63,1
4,d957021f1127559cd947b62533f484f7,0004aac84e0df4da2b147fca70cf8255,1,sorocaba,SP,2017-11-14 19:45:42,2017-11-27 23:08:56,13.0,2017-11,2017,180.00,16.89,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
98661,725cf8e9c24e679a8a5a32cb92c9ce1e,fffcf5a5ff07b0908bd4e2dbc735a684,1,sanharo,PE,2017-06-08 21:00:36,2017-07-06 09:22:00,28.0,2017-06,2017,1570.00,497.42,2
98662,c71b9252fd7b3b263aaa4cb09319a323,fffea47cd6d3cc0a88bd621562a9d061,1,feira de santana,BA,2017-12-10 20:07:56,2018-01-09 22:28:20,30.0,2017-12,2017,64.89,19.69,1
98663,fdc45e6c7555e6cb3cc0daca2557dbe1,ffff371b4d645b6ecea244b27531430a,1,sinop,MT,2017-02-07 15:49:16,2017-02-22 12:45:04,15.0,2017-02,2017,89.90,22.56,1
98664,94d3ee0bc2a0af9d4fa47a4d63616e8d,ffff5962728ec6157033ef9805bacc48,1,bom jesus do norte,ES,2018-05-02 15:17:41,2018-05-14 11:54:26,12.0,2018-05,2018,115.00,18.69,1


### Аналитические запросы

In [ ]:
### Запрос 1 - Категории товаров с самой большой выручкой (Топ 10)

connection = sqlite3.connect("olist.db")

# Пишем запрос в базу данных
query = """
select 
    pcn.product_category_name_english, 
    cast(sum(oi.price) as int) as total_sum
from order_items oi
left join products pr on oi.product_id=pr.product_id
left join product_category_name_translation pcn on pr.product_category_name=pcn.product_category_name
group by
    pcn.product_category_name_english
order by total_sum desc
limit 10
;
"""

# Получаем данные и сохраняем их в датафрейм
top_categories = pd.read_sql(query, connection)
connection.close()

top_categories

,product_category_name_english,total_sum
0,health_beauty,1258681
1,watches_gifts,1205005
2,bed_bath_table,1036988
3,sports_leisure,988048
4,computers_accessories,911954
5,furniture_decor,729762
6,cool_stuff,635290
7,housewares,632248
8,auto,592720
9,garden_tools,485256


In [ ]:
### Запрос 2 - Клиенты с самой большой суммой заказов и их локация (Топ 10)

connection = sqlite3.connect("olist.db")

# Пишем запрос в базу данных
query = """
select 
    c.customer_unique_id, 
    c.customer_city, 
    c.customer_state, 
    SUM(oi.price + oi.freight_value) as spending_sum
from customers c
left join orders o on c.customer_id = o.customer_id
left join order_items oi on o.order_id = oi.order_id
group by 
    c.customer_unique_id, 
    c.customer_city, 
    c.customer_state
order by spending_sum desc 
limit 10
;
"""

# Получаем данные и сохраняем их в датафрейм
top_clients_by_spendings = pd.read_sql(query, connection)
connection.close()

top_clients_by_spendings

,customer_unique_id,customer_city,customer_state,spending_sum
0,0a0a92112bd4c708ca5fde585afaa872,rio de janeiro,RJ,13664.08
1,da122df9eeddfedc1dc1f5349a1a690c,araruama,RJ,7571.63
2,763c8b1c9c68a0229c42c9fc6f662b93,vila velha,ES,7274.88
3,dc4802a71eae9be1dd28f5d788ceb526,campo grande,MS,6929.31
4,459bef486812aa25204be022145caa62,vitoria,ES,6922.21
5,ff4159b92c40ebe40454e3e6a7c35ed6,marilia,SP,6726.66
6,4007669dec559734d6f53e029e360987,divinopolis,MG,6081.54
7,5d0a2980b292d049061542014e8960bf,goiania,GO,4809.44
8,eebb5dda148d3893cdaf5b5ca3040ccb,maua,SP,4764.34
9,48e1ac109decbb87765a3eade6854098,joao pessoa,PB,4681.78


In [ ]:
### Запрос 3 - Города с самой большой суммой заказов (Топ 10)

connection = sqlite3.connect("olist.db")

# Пишем запрос в базу данных
query = """
select 
    c.customer_city, 
    c.customer_state, 
    SUM(oi.price+oi.freight_value) as spending_sum
from customers c
left join orders o on c.customer_id=o.customer_id
left join order_items oi on o.order_id=oi.order_id
group by c.customer_city, c.customer_state
order by spending_sum desc 
limit 10
;
"""

# Получаем данные и сохраняем их в датафрейм
top_cities_by_spendings = pd.read_sql(query, connection)
connection.close()

top_cities_by_spendings

,customer_city,customer_state,spending_sum
0,sao paulo,SP,2170227.12
1,rio de janeiro,RJ,1154234.02
2,belo horizonte,MG,416733.39
3,brasilia,DF,352305.14
4,curitiba,PR,244739.87
5,porto alegre,RS,224064.09
6,salvador,BA,216772.40
7,campinas,SP,212541.70
8,guarulhos,SP,163575.82
9,niteroi,RJ,137919.38


In [ ]:
### Запрос 4 - Динамика выручки с заказов в Sao Paulo по месяцам 

connection = sqlite3.connect("olist.db")

# Пишем запрос в базу данных
query = """
select 
    strftime('%Y-%m', o.order_purchase_timestamp) as year_month,
    sum(oi.price+oi.freight_value) as total_orders_value
from orders o
join customers c on o.customer_id=c.customer_id
join order_items oi on o.order_id=oi.order_id
where c.customer_city='sao paulo'
group by strftime('%Y-%m', o.order_purchase_timestamp)
order by year_month
;
"""

# Получаем данные и сохраняем их в датафрейм
monthly_orders_sao_paulo = pd.read_sql(query, connection)
connection.close()

monthly_orders_sao_paulo

,year_month,total_orders_value
0,2016-10,4607.50
1,2017-01,18014.55
2,2017-02,32250.02
3,2017-03,61421.48
4,2017-04,50742.49
5,2017-05,71788.40
6,2017-06,64655.96
7,2017-07,73321.53
8,2017-08,80278.44
9,2017-09,91440.14


In [ ]:
### Запрос 5 - Клиенты, количество и id их заказов без отзыва 

connection = sqlite3.connect("olist.db")

# Пишем запрос в базу данных
query = """
select 
   c.customer_unique_id,
   o.order_id,
   count(o.order_id) over (partition by c.customer_unique_id) as orders_without_review_cnt
from customers c 
join orders o on c.customer_id = o.customer_id
left join order_reviews orr on o.order_id = orr.order_id
where orr.review_id is null
;
"""

# Получаем данные и сохраняем их в датафрейм
review_customers = pd.read_sql(query, connection)
connection.close()

review_customers

,customer_unique_id,order_id,orders_without_review_cnt
0,001ae44fa04911a9e9577356dce6c63c,a59ef0abffbef8ddaae23600b6ee6604,1
1,00bee19e1199bc5cede1c674177b9e22,6147e0d51f6ecc3e0ae6defe7156bc82,1
2,00fb9cc86736fe3782dbc828a2eea4a6,c4dacba4560d22bf516c37d850225b2c,1
3,025cf7c2f32536f0be1ab412bb6602d2,a46e7e8e02915adecb392ad065121a51,1
4,02a2a0a9627a062602de754d75038ce4,f289bb2bb19d49b2e6aa64cb10b0e4bc,1
...,...,...,...
763,ff905cb313cb121a840dafe5a7a9170f,458a97b5d6f66560f0b625effac969a2,1
764,ffb3cbd5e1e507679e2db84c515410a5,ff5b7f440481674b38554434638beebf,1
765,ffca77d46c6d7d815549a5e4482dba2f,7d9dad1ac923f80a77b7a8fdd10a7201,1
766,ffe0c10afc687bcf34a0451f2b87dd9b,3c21823fb08a6a2ff770877efffb1c1a,1
